In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.models import Model, load_model
import os
import glob
from keras.datasets import fashion_mnist
from tensorflow.keras.initializers import GlorotUniform, HeNormal, RandomNormal
from numpy.random import randn, randint

In [ ]:
# Load Fashion MNIST dataset

(data, _), (_, _) = fashion_mnist.load_data()

In [ ]:
# A function to normalize data
def normalize_data(data):
    # having input values in [-1,1] improves the numerical stability of the optimization problem
    norm_data = (np.float32(data)/255. -0.5) *2.
    norm_data = np.clip(norm_data, -1, 1) # just a sanity check
    return norm_data

# A function to denormalize data
def denormalize_data(norm_data):
    data = 255*(norm_data + 1)/2
    return data

In [ ]:
# Normalize the data
norm_data = normalize_data(data)
print('Data shape:', data.shape)

print('Normalized Data shape:', norm_data.shape)
print(norm_data.dtype)

In [ ]:
# A function to sample a batch of real data
def sample_real_data(data, numdata):
    random_pos = np.random.choice(len(data), numdata, replace=False)
    X = data[random_pos]
    y = np.ones((numdata, 1)) # the default label for real data is one
    return X, y, random_pos

In [ ]:
# a function to visualize a batch of data
def visualizeSamples(data, pos):
    numdata = len(pos)
    ncols = 5
    nrows = int(numdata/ncols);

    fig, axes = plt.subplots(nrows, ncols, figsize=(5, 5))

    k = 0
    for i in range(4):
        for j in range(5):

            axes[i, j].imshow(data[k,:,:], cmap='gray')
            axes[i, j].axis('off')
            axes[i, j].set_title(f"{pos[k]}")
            k = k+1

    plt.tight_layout()
    plt.show()

In [ ]:
# Select some random data to be used as a reference in the visualization
numdata = 20
samp_data, samp_label, samp_pos = sample_real_data(data, numdata)
# Visualize selected input images
visualizeSamples(samp_data, samp_pos)

In [ ]:
# Build Autoencoder
data_conv = np.expand_dims(norm_data, axis=-1)  # Add channel for convolutions
input_shape = data_conv.shape[1:]
print(input_shape)

encoder = keras.Sequential([
    layers.Input(shape=input_shape),
    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2), padding="same"),
    layers.Conv2D(16, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2), padding="same")
])

encoder.summary()
decoder = keras.Sequential([
    layers.Input(shape=(7,7,16)),
    layers.Conv2DTranspose(16, (3, 3), activation="relu", padding="same"),
    layers.UpSampling2D((2, 2)),
    layers.Conv2DTranspose(32, (3, 3), activation="relu", padding="same"),
    layers.UpSampling2D((2, 2)),
    layers.Conv2D(1, (3, 3), activation="tanh", padding="same")
])
decoder.summary()

autoencoder = keras.Sequential([encoder, decoder])


In [ ]:
# Compile and train Autoencoder
autoencoder.compile(optimizer="adam", loss="mse")
history = autoencoder.fit(norm_data, norm_data, epochs=20, batch_size=16, validation_split=0.2)

In [ ]:
# Plot training and validation loss
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Autoencoder Training Loss')
plt.legend()
plt.show()

In [ ]:
# Generate reconstructed images
samp_data_norm = normalize_data(samp_data)
samp_data_norm = np.expand_dims(samp_data_norm, axis=-1)
encoded_imgs = encoder.predict(samp_data_norm)
decoded_imgs = decoder.predict(encoded_imgs)

print(np.shape(decoded_imgs))

decoded_imgs = denormalize_data(decoded_imgs)

# Visualize selected input images
visualizeSamples(decoded_imgs, samp_pos)

In [ ]:
# Try and generate new data with the decoder
# To this purpose we need to start from random noise

random_noise = np.random.rand(numdata, 32, 54, 16)  # It must fit the input of the decoder

# Generate reconstructed images
generated_imgs = decoder.predict(random_noise)

print("Shape of generate data: ", np.shape(generated_imgs))

generated_imgs = denormalize_data(generated_imgs)

# Visualize selected input images
pos = np.arange(numdata)
visualizeSamples(generated_imgs, pos)


In [ ]:
# Build a Variational Autoencoder
data_conv = np.expand_dims(norm_data, axis=-1)  # Add channel for convolutions
input_shape = data_conv.shape[1:]
print(input_shape)

latent_dim = 16

# Sampling layer
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        epsilon = tf.random.normal(shape=tf.shape(z_mean))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

# -------------------------
# Encoder
# -------------------------
encoder_inputs = keras.Input(shape=input_shape)

x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(encoder_inputs)
x = layers.MaxPooling2D((2, 2), padding="same")(x)
x = layers.Conv2D(16, (3, 3), activation="relu", padding="same")(x)
x = layers.MaxPooling2D((2, 2), padding="same")(x)

shape_before_flatten = keras.backend.int_shape(x)[1:]   # ad es. (7, 7, 16)

x = layers.Flatten()(x)
x = layers.Dense(64, activation="relu")(x)

z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])

encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")
encoder.summary()

# -------------------------
# Decoder
# -------------------------
latent_inputs = keras.Input(shape=(latent_dim,))

x = layers.Dense(np.prod(shape_before_flatten), activation="relu")(latent_inputs)
x = layers.Reshape(shape_before_flatten)(x)

x = layers.Conv2DTranspose(16, (3, 3), activation="relu", padding="same")(x)
x = layers.UpSampling2D((2, 2))(x)
x = layers.Conv2DTranspose(32, (3, 3), activation="relu", padding="same")(x)
x = layers.UpSampling2D((2, 2))(x)
decoder_outputs = layers.Conv2D(1, (3, 3), activation="tanh", padding="same")(x)

decoder = keras.Model(latent_inputs, decoder_outputs, name="decoder")
decoder.summary()

# -------------------------
# VAE model
# -------------------------
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)

        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(tf.square(inputs - reconstruction), axis=(1, 2, 3))
        )

        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                axis=1
            )
        )

        self.add_loss(reconstruction_loss + kl_loss)
        return reconstruction

vae = VAE(encoder, decoder)
vae.compile(optimizer="adam")

In [ ]:
vae.fit(data_conv, data_conv, epochs=20, batch_size=128)

In [ ]:
# Generate reconstructed images
samp_data_norm = normalize_data(samp_data)
samp_data_norm = np.expand_dims(samp_data_norm, axis=-1)

z_mean, z_log_var, z = encoder.predict(samp_data_norm)
decoded_imgs = decoder.predict(z)

print(np.shape(decoded_imgs))

decoded_imgs = denormalize_data(decoded_imgs)

# Remove channel dimension if needed
decoded_imgs = np.squeeze(decoded_imgs, axis=-1)

# Visualize selected input images
visualizeSamples(decoded_imgs, samp_pos)

In [ ]:
# Build a GAN composed of fully connected layers
def build_generator(latent_dim):
    model = tf.keras.Sequential([

        layers.Input(shape=(latent_dim,)),
        layers.Dense(256),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dense(512),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dense(1024),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dense(28*28*1, activation="tanh"),
        layers.Reshape((28,28,1))

    ])
    return model

def build_discriminator(input_shape):
    model = keras.Sequential([

        layers.Input(shape=input_shape),
        layers.Flatten(),

        layers.Dense(1024),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Dense(512),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Dense(256),
        layers.LeakyReLU(negative_slope=0.2),
        layers.Dropout(0.3),

        layers.Dense(1, activation="sigmoid")

    ])

    opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])\

    return model

In [ ]:
# Build the generator and discriminator models
latent_dim = 5
input_shape = (28, 28, 1)

print("Input shape for discriminator:", input_shape)

discriminator = build_discriminator(input_shape)

print("DISCRIMINATOR")
discriminator.summary()

generator = build_generator(latent_dim)

print("GENERATOR")
generator.summary()

In [ ]:
# Build the GAN
def define_gan(g_model, d_model, latent_dim):
    d_model.trainable = False # This means that when training the generator, the discriminator is not trained

    # Define the input for the GAN (which is the input for the generator)
    gan_input = keras.Input(shape=(latent_dim,))

    # Connect the generator to this input
    generated_image = g_model(gan_input)

    # Connect the discriminator to the generated image
    gan_output = d_model(generated_image)

    # Create the GAN model
    gan_model = keras.Model(gan_input, gan_output)

    opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
    gan_model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    return gan_model

# Explicitly build the generator and discriminator by calling them with dummy inputs
# This ensures their .input and .output attributes are populated before define_gan is called.
dummy_latent_input = tf.zeros((1, latent_dim))
_ = generator(dummy_latent_input) # Call generator to build it

dummy_image_input = tf.zeros((1, *input_shape))
_ = discriminator(dummy_image_input) # Call discriminator to build it

gan_model = define_gan(generator, discriminator, latent_dim)
gan_model.summary()

In [ ]:
# A function to generate random inputs for the decoder
def generate_latent_points(latent_dim, n_samples):
    x = randn(latent_dim * n_samples)
    z = x.reshape(n_samples, latent_dim)
    return z

# A function to generate a fake input from a random point
def generate_fake_samples(generator, latent_dim, n_samples):
    z = generate_latent_points(latent_dim, n_samples)
    generated_imgs = generator.predict(z)
    y = np.zeros((n_samples, 1)) # the default label for fake data is 0
    return generated_imgs, y

In [ ]:
# Some functions to check the performance

# To save an intermediate model
def summarize_performance(step, g_model):
    filename = 'model_%04d.h5' % (step+1)
    g_model.save(filename)
    print('>Saved: %s' % (filename))

# To plot the loss ofdiscriminator and generator
def plot_loss(d_loss, g_loss, dim):
    plt.close()
    plt.figure(figsize=dim)
    plt.plot(d_loss)
    plt.plot(g_loss)
    plt.title('Model loss')
    plt.ylabel('Loss')
    plt.xlabel('Iteration')
    plt.legend(['Discriminator', 'Generator'], loc='upper right')
    plt.show()

# To plot the accuracy ofdiscriminator and generator
def plot_acc(d_acc,g_acc, dim):
    plt.close()
    plt.figure(figsize=dim)
    plt.plot(d_acc)
    plt.plot(g_acc)
    plt.title('Model accuracy')
    plt.ylabel('Accuracy')
    plt.xlabel('Iteration')
    plt.legend(['Discriminator', 'Generator'], loc='upper right')
    plt.show()

In [ ]:
def train(g_model, d_model, gan_model, X_train, latent_dim, n_epochs=100, n_batch=64):

    bat_per_epo = int(X_train.shape[0] / n_batch)
    n_steps = bat_per_epo * n_epochs

    d_loss_vec = []
    g_loss_vec = []

    d_acc_vec = [];
    g_acc_vec = [];

    for i in range(n_steps):

        d_loss_b = []
        g_loss_b = []

        d_acc_b = [];
        g_acc_b = [];

        X_real, y_real,_ = sample_real_data(X_train, n_batch) # sample real data

        print(np.shape(X_real))
        X_fake, y_fake = generate_fake_samples(g_model, latent_dim, n_batch) # generate fake data

        # Training the discriminator
        d_loss_r, d_acc_r = d_model.train_on_batch(X_real, y_real) # training step on real data
        d_loss_f, d_acc_f = d_model.train_on_batch(X_fake, y_fake) # training step on fake data

        d_loss_b.append((d_loss_r + d_loss_f)/2)
        d_acc_b.append((d_acc_r + d_acc_f)/2)

        # Training the generator (i.e. training the GAN)
        z_input = generate_latent_points(latent_dim, n_batch)
        y_gan = np.ones((n_batch, 1))
        g_loss, g_acc = gan_model.train_on_batch(z_input, y_gan)

        g_loss_b.append(g_loss)
        g_acc_b.append(g_acc)

        print('>%d, D loss-acc [%.3f,%.3f], G/GAN loss-acc [%.3f,%.3f]' % (i+1, (d_loss_r + d_loss_f)/2, (d_loss_f + d_loss_f)/2, g_loss,g_acc))

        if (i+1) % (bat_per_epo * 1) == 0:

            summarize_performance(i, g_model)

            d_loss_vec.append(np.mean(d_loss_b))
            g_loss_vec.append(np.mean(g_loss_b))
            d_acc_vec.append(np.mean(d_acc_b))
            g_acc_vec.append(np.mean(g_acc_b))

            d_loss_b = []
            g_loss_b = []

            d_acc_b = [];
            g_acc_b = [];

            plot_loss(d_loss_vec,g_loss_vec, (6,4))
            plot_acc(d_acc_vec,g_acc_vec, (6,4))


In [ ]:
# Train the GAN model
train(generator, discriminator, gan_model, norm_data, latent_dim, n_epochs=10, n_batch=128)

In [ ]:
# Try and generate some samples with the GAN
n_examples = 20

latent_points = generate_latent_points(latent_dim, n_examples)
generated_imgs  = generator.predict(latent_points)
print(np.shape(generated_imgs))

generated_imgs = denormalize_data(generated_imgs)
pos = np.arange(n_examples)
visualizeSamples(generated_imgs, pos)


In [ ]:
# Build a GAN composed of convolutional layers
def build_generator(latent_dim):
    model = tf.keras.Sequential([


        layers.Dense(7 * 7 * 16, input_dim=latent_dim),
        layers.Reshape((7, 7, 16)),
        layers.Conv2DTranspose(16, (3, 3), activation="tanh", padding="same"),
        layers.UpSampling2D((2, 2)),
        layers.Conv2DTranspose(32, (3, 3), activation="tanh", padding="same"),
        layers.UpSampling2D((2, 2)),
        layers.Conv2D(1, (3, 3), activation="tanh", padding="same")

    ])
    return model

def build_discriminator(input_shape):
    model = keras.Sequential([

        layers.Input(shape=input_shape),
        layers.Flatten(),

        layers.Dense(1024),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),

        layers.Dense(512),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),

        layers.Dense(256),
        layers.LeakyReLU(alpha=0.2),
        layers.Dropout(0.3),

        layers.Dense(1, activation="sigmoid")

    ])

    opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])\

    return model


In [ ]:
latent_dim = 5
input_shape = (28, 28, 1)

print("Input shape for discriminator:", input_shape)

discriminator = build_discriminator(input_shape)

print("DISCRIMINATOR")
discriminator.summary()

generator = build_generator(latent_dim)

print("GENERATOR")
generator.summary()

In [ ]:
def define_gan(g_model, d_model, latent_dim):
    d_model.trainable = False # This means that when training the generator, the discriminator is not trained

    # Define the input for the GAN (which is the input for the generator)
    gan_input = keras.Input(shape=(latent_dim,))

    # Connect the generator to this input
    generated_image = g_model(gan_input)

    # Connect the discriminator to the generated image
    gan_output = d_model(generated_image)

    # Create the GAN model
    gan_model = keras.Model(gan_input, gan_output)

    opt = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
    gan_model.compile(loss='binary_crossentropy', optimizer=opt, metrics=['accuracy'])
    return gan_model

# Explicitly build the generator by calling it with a dummy input (for standalone use like summary/prediction)
generator(tf.zeros((1, latent_dim)))
# Explicitly build the discriminator by calling it with a dummy input (for standalone use like summary/prediction)
discriminator(tf.zeros((1, *input_shape)))

gan_model = define_gan(generator, discriminator, latent_dim)
gan_model.summary()

In [ ]:
# Train the GAN model
train(generator, discriminator, gan_model, norm_data, latent_dim, n_epochs=10, n_batch=128)

In [ ]:
# Try and generate some samples with the GAN
n_examples = 20

latent_points = generate_latent_points(latent_dim, n_examples)
generated_imgs  = generator.predict(latent_points)
print(np.shape(generated_imgs))

generated_imgs = denormalize_data(generated_imgs)
pos = np.arange(n_examples)
visualizeSamples(generated_imgs, pos)